# Worker-Based Execution

In [1]:
import openfe

load an AlchemicalNetwork to execute - this is a small example campaign with only two edges and pre-charged ligands

In [3]:
alchemical_network = openfe.AlchemicalNetwork.from_json("alchemicalNetwork_mc1_small.json")

In [4]:
# build a database of tasks from the AlchemicalNetwork
from openfe.orchestration import build_task_db_from_alchemical_network

campaign_name = "mcl1"
task_db, wh = build_task_db_from_alchemical_network(
    alchemical_network=alchemical_network,
    warehouse_dir=campaign_name,
    db_path=f"{campaign_name}.db"
)

In [5]:
task_db

TaskStatusDB(engine=Engine(sqlite:///mcl1.db))

In [6]:
# verify that the alchemicalnetwork is stored
wh.exists(alchemical_network.key)

True

In [7]:
from openfe.orchestration import get_task_df, get_dependency_df

task_df = get_task_df(task_db);
task_df 

,taskid,status,last_modified,tries,max_tries,task_type
0,HybridTopologySetupUnit-ecf65f3541c5437bb72715...,AVAILABLE,NaT,0,1,
1,HybridTopologySetupUnit-355765327d874fdda08b98...,AVAILABLE,NaT,0,1,
2,HybridTopologySetupUnit-3e15a145d9be4c9aa58d27...,AVAILABLE,NaT,0,1,
3,HybridTopologySetupUnit-17736cf42a85408f907ff6...,AVAILABLE,NaT,0,1,
4,HybridTopologyMultiStateSimulationUnit-2618e46...,BLOCKED,NaT,0,1,
5,HybridTopologyMultiStateSimulationUnit-5971e55...,BLOCKED,NaT,0,1,
6,HybridTopologyMultiStateSimulationUnit-2e04387...,BLOCKED,NaT,0,1,
7,HybridTopologyMultiStateSimulationUnit-d96090d...,BLOCKED,NaT,0,1,
8,HybridTopologyMultiStateAnalysisUnit-f5b1efe10...,BLOCKED,NaT,0,1,
9,HybridTopologyMultiStateAnalysisUnit-69e78e802...,BLOCKED,NaT,0,1,


In [8]:
dep_df = get_dependency_df(task_db);
dep_df

,from,to,blocking
0,HybridTopologySetupUnit-ecf65f3541c5437bb72715...,HybridTopologyMultiStateSimulationUnit-2618e46...,True
1,HybridTopologySetupUnit-355765327d874fdda08b98...,HybridTopologyMultiStateSimulationUnit-5971e55...,True
2,HybridTopologySetupUnit-3e15a145d9be4c9aa58d27...,HybridTopologyMultiStateSimulationUnit-2e04387...,True
3,HybridTopologySetupUnit-17736cf42a85408f907ff6...,HybridTopologyMultiStateSimulationUnit-d96090d...,True
4,HybridTopologySetupUnit-ecf65f3541c5437bb72715...,HybridTopologyMultiStateAnalysisUnit-f5b1efe10...,True
5,HybridTopologyMultiStateSimulationUnit-2618e46...,HybridTopologyMultiStateAnalysisUnit-f5b1efe10...,True
6,HybridTopologySetupUnit-355765327d874fdda08b98...,HybridTopologyMultiStateAnalysisUnit-69e78e802...,True
7,HybridTopologyMultiStateSimulationUnit-5971e55...,HybridTopologyMultiStateAnalysisUnit-69e78e802...,True
8,HybridTopologySetupUnit-3e15a145d9be4c9aa58d27...,HybridTopologyMultiStateAnalysisUnit-e1926eb0c...,True
9,HybridTopologyMultiStateSimulationUnit-2e04387...,HybridTopologyMultiStateAnalysisUnit-e1926eb0c...,True


## use a Worker to run the jobs

In [9]:
from openfe.orchestration import Worker
db_path = f"{wh.name}.db"
worker = Worker(wh, db_path)

In [10]:
# run this cell to execute units and do a status check
n_submissions = 4
for _ in range(n_submissions):
    exec_result = worker.execute_unit(scratch='scratch')
    print(exec_result)

INFO:openfe.utils.system_probe.log:SYSTEM CONFIG DETAILS:
INFO:openfe.utils.system_probe.log.hostname:hostname: 'Alyssas-Macbook-Pro.local'
INFO:openfe.utils.system_probe.log.gpu:CUDA-based GPU not found
INFO:openfe.utils.system_probe.log:Memory used: 21.4G (62.7%)
INFO:openfe.utils.system_probe.log:scratch/task_workdirs/HybridTopologySetupUnit-ecf65f3541c5437bb727155d0c2692a0: 66% full (312.2G free)
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Starting system setup unit
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Parameterizing systems
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 0 (LIG) of chain 0>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][C]1=[C]([C](=[O])[O-])[S][c]2[c]([H])[c]([H])[c]([H])[c]([H])[c]21 using openff-2.2.1
INFO:exorcist.taskdb:Marking try of HybridTopologySetupUnit-ecf65f3541c5437bb7

('HybridTopologySetupUnit-ecf65f3541c5437bb727155d0c2692a0', ProtocolUnitFailure(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 153 (LIG) of chain 1>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][C]1=[C]([C](=[O])[O-])[S][c]2[c]([H])[c]([H])[c]([H])[c]([H])[c]21 using openff-2.2.1
INFO:exorcist.taskdb:Marking try of HybridTopologySetupUnit-355765327d874fdda08b98c1f99b4b34 as failed.
INFO:exorcist.taskdb:Checking out a new task
INFO:exorcist.taskdb:Selected task 'HybridTopologySetupUnit-3e15a145d9be4c9aa58d270f7e06df54'
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial ch

('HybridTopologySetupUnit-355765327d874fdda08b98c1f99b4b34', ProtocolUnitFailure(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 0 (LIG) of chain 0>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:exorcist.taskdb:Marking try of HybridTopologySetupUnit-3e15a145d9be4c9aa58d270f7e06df54 as failed.
INFO:exorcist.taskdb:Checking out a new task
INFO:exorcist.taskdb:Selected task 'HybridTopologySetupUnit-17736cf42a85408f907ff64fc78a9b72'
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charg

('HybridTopologySetupUnit-3e15a145d9be4c9aa58d270f7e06df54', ProtocolUnitFailure(HybridTopology Setup: ligand_2 to ligand_3 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 153 (LIG) of chain 1>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:exorcist.taskdb:Marking try of HybridTopologySetupUnit-17736cf42a85408f907ff64fc78a9b72 as failed.


('HybridTopologySetupUnit-17736cf42a85408f907ff64fc78a9b72', ProtocolUnitFailure(HybridTopology Setup: ligand_2 to ligand_3 repeat 0 generation 0))


In [11]:
df=get_task_df(task_db)
df

,taskid,status,last_modified,tries,max_tries,task_type
0,HybridTopologySetupUnit-ecf65f3541c5437bb72715...,TOO_MANY_RETRIES,2026-08-27 15:16:23.081446,1,1,
1,HybridTopologySetupUnit-355765327d874fdda08b98...,TOO_MANY_RETRIES,2026-08-27 15:16:23.668511,1,1,
2,HybridTopologySetupUnit-3e15a145d9be4c9aa58d27...,TOO_MANY_RETRIES,2026-08-27 15:16:24.056751,1,1,
3,HybridTopologySetupUnit-17736cf42a85408f907ff6...,TOO_MANY_RETRIES,2026-08-27 15:16:24.439276,1,1,
4,HybridTopologyMultiStateSimulationUnit-2618e46...,BLOCKED,NaT,0,1,
5,HybridTopologyMultiStateSimulationUnit-5971e55...,BLOCKED,NaT,0,1,
6,HybridTopologyMultiStateSimulationUnit-2e04387...,BLOCKED,NaT,0,1,
7,HybridTopologyMultiStateSimulationUnit-d96090d...,BLOCKED,NaT,0,1,
8,HybridTopologyMultiStateAnalysisUnit-f5b1efe10...,BLOCKED,NaT,0,1,
9,HybridTopologyMultiStateAnalysisUnit-69e78e802...,BLOCKED,NaT,0,1,


## Load results 
(cooking show magic)

In [14]:
# load a different warehouse
from openfe.storage.warehouse import FileSystemWarehouse
other_wh = FileSystemWarehouse.from_dir("mcl1")

In [15]:
result_edges = other_wh.gather_all_results();

INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e5a4596e9d800367c6832564a93877f3 (name: 'ligand_1')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e5a4596e9d800367c6832564a93877f3 (name: 'ligand_1')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges

In [16]:
result_edges

[(<RelativeHybridTopologyProtocolResult-8301d203efc373156a71d2307c2b45b4>,
  <ProtocolDAGResult-33d7e566301e554d5d3cf8a3c02cfdc7>),
 (<RelativeHybridTopologyProtocolResult-8301d203efc373156a71d2307c2b45b4>,
  <ProtocolDAGResult-91f17463e711af5197c03f15e02c7b0f>),
 (<RelativeHybridTopologyProtocolResult-8301d203efc373156a71d2307c2b45b4>,
  <ProtocolDAGResult-e234027ae25acb45a1399386a820a025>),
 (<RelativeHybridTopologyProtocolResult-8301d203efc373156a71d2307c2b45b4>,
  <ProtocolDAGResult-d190ced937850252a6ccecea13014c79>)]

In [17]:
[(pr.get_estimate(), pr.get_uncertainty()) for pr, dag in result_edges if dag.ok()]

[]